# External connectivity: a local MCP calculator

Copyright (c) Patrick Hall (jphall@gwu.edu), 2026-27. Released under the MIT License; see [SOFTWARE_LICENSE.txt](../../SOFTWARE_LICENSE.txt).

Discover a calculator on a **separate local server process**, expose its contract to the model, invoke it through MCP, and return the result for a final answer. The notebook is the host, `ClientSession` is the client, and `calculator_server.py` is the server.

Run the cells in order using the locked project environment. The notebook starts and stops its own local server over stdio; do not start the server separately. Only the host calls the OpenAI API and receives the API key.

See the [Lecture 6 README](README.md) for `uv`, VS Code, and JupyterLab setup, the optional standalone-server command, and the MCP SDK version constraint.

## 1. Configure the host
Use the same project-root credential loading and model setting as the Text2SQL notebook.

In [1]:
# 1. Configure the direct OpenAI client
# Imports
import os
import sys
import json
from pathlib import Path
from getpass import getpass
from time import monotonic
from dotenv import load_dotenv
from openai import OpenAI

# Project paths and credential loading
# Find the project by its configuration, even if .env has not been created yet.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file())
lecture_dir = project_root / 'lecture' / 'lec_6'
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')

# Bounded synchronous OpenAI client
client = OpenAI(api_key=api_key, timeout=45, max_retries=1)
MODEL = os.getenv('OPENAI_TEXT_MODEL', 'gpt-5-mini')
print('Model:', MODEL)  # Never print credentials.

Model: gpt-5-mini


## 2. Inspect the separate server code

The server exports two safe, typed operations: `revenue_growth` computes a percentage, while `sort_numbers` is an intentionally irrelevant alternative for this question. Both validate their inputs and return structured data; neither evaluates arbitrary Python expressions. stdout is reserved for MCP messages; logs belong on stderr.

Sources: [MCP Python SDK v1](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x), [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture).

In [2]:
# 2. Display the exact server implementation used by the subprocess
server_path = lecture_dir / 'calculator_server.py'
print(server_path.read_text())

"""Separate local MCP calculator server. Copyright Patrick Hall, 2026-27. MIT License."""
import math
import os
from mcp.server.fastmcp import FastMCP

# The calculator has no need to inherit the host's API credential.
os.environ.pop('OPENAI_API_KEY', None)
mcp = FastMCP('Lecture 6 calculator')


# Side-effect-free tool definitions.
@mcp.tool()
def revenue_growth(previous: float, current: float) -> dict:
    """Return percentage revenue growth: 100 * (current - previous) / previous.

    Inputs must be finite, nonnegative revenues in the same currency and scale;
    previous must be positive. This performs arithmetic only, with no side effects.
    """
    if not math.isfinite(previous) or not math.isfinite(current) or previous <= 0 or current < 0:
        raise ValueError('Use finite revenues, previous > 0 and current >= 0.')
    return {'previous': previous, 'current': current,
            'growth_percent': 100 * (current - previous) / previous}


@mcp.tool()
def sort_numbers(values:

## 3. Discover, request, execute, and answer

`stdio_client` launches the server using this kernel's Python interpreter and cleans it up when its context closes. After initialization, `list_tools()` discovers both tool contracts. The host allowlists and exposes both, then displays the model's request **before execution**, validates the selected tool and arguments locally, and sends the approved call with `call_tool`.

For this worked example, `tool_choice='required'` requires one tool request, and `parallel_tool_calls=False` keeps the handoff visible. The host checks that the model selects `revenue_growth`, not the irrelevant `sort_numbers`. One successful choice illustrates routing; it does not establish general tool-selection reliability. The final call receives the observation and no tools. This is a fixed two-request workflow, not a general-purpose autonomous agent. Notebook top-level `await` works with Jupyter's existing event loop; do not wrap it in `asyncio.run()`.

Source: [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling).

In [3]:
# 3. Launch the server, discover its contract, and execute one live model request
# Async, MCP, and validation imports
import asyncio
import math
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import BaseModel, ConfigDict, Field

# Host-side argument models and explicit tool allowlist
class GrowthArgs(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True, allow_inf_nan=False)
    previous: float = Field(gt=0)
    current: float = Field(ge=0)

class SortArgs(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True, allow_inf_nan=False)
    values: list[float] = Field(min_length=1, max_length=20)
    descending: bool

APPROVED_TOOLS = {'revenue_growth': GrowthArgs, 'sort_numbers': SortArgs}

async def calculator_demo():
    # Server configuration
    # Explicitly pass a small environment: the server never receives the API key.
    env = {k:v for k,v in os.environ.items() if k in
           {'PATH','SYSTEMROOT','WINDIR','TEMP','TMP','LANG'}}
    parameters = StdioServerParameters(command=sys.executable, args=[str(server_path)], env=env)

    # MCP session initialization
    async with stdio_client(parameters) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Tool discovery and contract construction
            discovered = await session.list_tools()
            discovered_by_name = {tool.name: tool for tool in discovered.tools}
            missing = APPROVED_TOOLS.keys() - discovered_by_name.keys()
            if missing:
                raise RuntimeError(f'Approved tools not discovered: {sorted(missing)}')
            print('DISCOVERED:', discovered.model_dump_json(indent=2))

            # Expose only explicitly approved discovered tools, not every server capability.
            contracts = []
            for name in APPROVED_TOOLS:
                tool = discovered_by_name[name]
                schema = dict(tool.inputSchema)
                schema['additionalProperties'] = False
                contracts.append({'type':'function', 'name':name,
                                  'description':tool.description,
                                  'parameters':schema, 'strict':True})

            # Model tool request
            question = 'Find percentage revenue growth from $1.2 million to $1.5 million.'
            request = await asyncio.to_thread(
                client.responses.create, model=MODEL, input=question, tools=contracts,
                tool_choice='required', parallel_tool_calls=False, max_output_tokens=1800,
                reasoning={'effort':'low'}, store=False)
            if request.status != 'completed':
                raise RuntimeError(f'Incomplete model request: {request.status}')

            # Model request validation and tool-selection check
            calls = [x for x in request.output if x.type=='function_call']
            if len(calls) != 1:
                raise RuntimeError(f'Expected one tool request, received {len(calls)}.')
            call = calls[0]
            print('MODEL REQUEST:', call.name, call.arguments)
            if call.name not in APPROVED_TOOLS:
                raise RuntimeError(f'Unapproved tool requested: {call.name}')
            assert call.name == 'revenue_growth', f'Wrong tool selected: {call.name}'

            # Argument validation and dispatch
            args = APPROVED_TOOLS[call.name].model_validate_json(call.arguments)
            result = await session.call_tool(call.name, arguments=args.model_dump())
            print('MCP RESULT:', result.model_dump_json(indent=2))
            if result.isError:
                raise RuntimeError('Calculator tool failed; do not request an unsupported final answer.')

            # Result validation
            # Check both modern structured output and the v1 text-content representation.
            payload = result.structuredContent
            if payload is None:
                payload = json.loads(next(c.text for c in result.content if c.type=='text'))
            assert math.isclose(payload['growth_percent'], 25.0)
            assert math.isclose((args.current-args.previous)/args.previous, 0.25)

            # Final model response
            # Make a second model call after the calculator has returned validated evidence.
            # Run the synchronous OpenAI SDK call in a thread so it does not block MCP I/O.
            # Replay the question and original function request, then match the result by call_id.
            # No tools are supplied here: this call must explain the result, not request another action.
            final = await asyncio.to_thread(
                client.responses.create, model=MODEL,
                instructions='Answer only from the calculator result. State the percentage and denominator.',
                input=[{'role':'user','content':question}, *request.output,
                       {'type':'function_call_output','call_id':call.call_id,
                        'output':json.dumps(payload)}],
                max_output_tokens=1200, reasoning={'effort':'low'}, store=False)
            if final.status != 'completed' or not final.output_text:
                raise RuntimeError('The model did not produce a completed final answer.')

            print('FINAL ANSWER:', final.output_text)
            print('MODEL USAGE:', request.usage.model_dump(), final.usage.model_dump())

            # Negative server test
            # A server-side error is observable protocol data, not a successful calculation.
            invalid = await session.call_tool('revenue_growth', arguments={'previous':0,'current':1.5})
            assert invalid.isError
            print('ZERO-DENOMINATOR CHECK: rejected by the server')

            # Return value
            return {'growth_percent':payload['growth_percent'], 'answer':final.output_text,
                    'tool_name':call.name}

# Bounded demo invocation
# Bound protocol/model work as well as individual HTTP requests; context managers clean up.
mcp_demo = await asyncio.wait_for(calculator_demo(), timeout=120)
assert math.isclose(mcp_demo['growth_percent'], 25.0)

DISCOVERED: {
  "meta": null,
  "nextCursor": null,
  "tools": [
    {
      "name": "revenue_growth",
      "title": null,
      "description": "Return percentage revenue growth: 100 * (current - previous) / previous.\n\nInputs must be finite, nonnegative revenues in the same currency and scale;\nprevious must be positive. This performs arithmetic only, with no side effects.\n",
      "inputSchema": {
        "properties": {
          "previous": {
            "title": "Previous",
            "type": "number"
          },
          "current": {
            "title": "Current",
            "type": "number"
          }
        },
        "required": [
          "previous",
          "current"
        ],
        "title": "revenue_growthArguments",
        "type": "object"
      },
      "outputSchema": null,
      "icons": null,
      "annotations": null,
      "meta": null,
      "execution": null
    },
    {
      "name": "sort_numbers",
      "title": null,
      "description": "Sort 

The arithmetic result is **25%**: `(1.5 − 1.2) / 1.2 × 100`. Input revenues must use the same units. Given two approved tools, the model requested `revenue_growth`; the host validated and routed it; the server calculated; the host returned the observation; the model explained the result.

MCP standardizes discovery and invocation, not business authority or answer correctness. The host still approves tool names, limits calls, validates inputs, and controls credentials. A local stdio server is a trusted program running with the host's OS privileges; production isolation and remote-server authentication require additional design. This notebook does not demonstrate remote HTTP hosting.

Sources: [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture), [official MCP Python SDK v1 documentation](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x), [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling).